У цьому ДЗ ми потренуємось розв'язувати задачу багатокласової класифікації за допомогою логістичної регресії з використанням стратегій One-vs-Rest та One-vs-One, оцінити якість моделей та порівняти стратегії.

### Опис задачі і даних

**Контекст**

В цьому ДЗ ми працюємо з даними про сегментацію клієнтів.

Сегментація клієнтів – це практика поділу бази клієнтів на групи індивідів, які схожі між собою за певними критеріями, що мають значення для маркетингу, такими як вік, стать, інтереси та звички у витратах.

Компанії, які використовують сегментацію клієнтів, виходять з того, що кожен клієнт є унікальним і що їхні маркетингові зусилля будуть більш ефективними, якщо вони орієнтуватимуться на конкретні, менші групи зі зверненнями, які ці споживачі вважатимуть доречними та які спонукатимуть їх до купівлі. Компанії також сподіваються отримати глибше розуміння уподобань та потреб своїх клієнтів з метою виявлення того, що кожен сегмент цінує найбільше, щоб точніше адаптувати маркетингові матеріали до цього сегменту.

**Зміст**.

Автомобільна компанія планує вийти на нові ринки зі своїми існуючими продуктами (P1, P2, P3, P4 і P5). Після інтенсивного маркетингового дослідження вони дійшли висновку, що поведінка нового ринку схожа на їхній існуючий ринок.

На своєму існуючому ринку команда з продажу класифікувала всіх клієнтів на 4 сегменти (A, B, C, D). Потім вони здійснювали сегментовані звернення та комунікацію з різними сегментами клієнтів. Ця стратегія працювала для них надзвичайно добре. Вони планують використати ту саму стратегію на нових ринках і визначили 2627 нових потенційних клієнтів.

Ви маєте допомогти менеджеру передбачити правильну групу для нових клієнтів.

В цьому ДЗ використовуємо дані `customer_segmentation_train.csv`[скачати дані](https://drive.google.com/file/d/1VU1y2EwaHkVfr5RZ1U4MPWjeflAusK3w/view?usp=sharing). Це `train.csv`з цього [змагання](https://www.kaggle.com/datasets/abisheksudarshan/customer-segmentation/data?select=train.csv)

**Завдання 1.** Завантажте та підготуйте датасет до аналізу. Виконайте обробку пропущених значень та необхідне кодування категоріальних ознак. Розбийте на тренувальну і тестувальну вибірку, де в тесті 20%. Памʼятаємо, що весь препроцесинг ліпше все ж тренувати на тренувальній вибірці і на тестувальній лише використовувати вже натреновані трансформери.
Але в даному випадку оскільки значень в категоріях небагато, можна зробити обробку і на оригінальних даних, а потім розбити - це простіше. Можна також реалізувати процесинг і тренування моделі з пайплайнами. Обирайте як вам зручніше.

In [96]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder, FunctionTransformer, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.multiclass import OneVsRestClassifier
from imblearn.over_sampling import SMOTE, SMOTENC
from imblearn.combine import SMOTETomek

In [20]:
raw_df = pd.read_csv('customer_segmentation_train.csv')

In [26]:
raw_df.head()

,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
0,462809,Male,No,22,No,Healthcare,1.0,Low,4.0,Cat_4,D
1,462643,Female,Yes,38,Yes,Engineer,NaN,Average,3.0,Cat_4,A
2,466315,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,Cat_6,B
3,461735,Male,Yes,67,Yes,Lawyer,0.0,High,2.0,Cat_6,B
4,462669,Female,Yes,40,Yes,Entertainment,NaN,High,6.0,Cat_6,A


In [27]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8068 entries, 0 to 8067
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ID               8068 non-null   int64  
 1   Gender           8068 non-null   object 
 2   Ever_Married     7928 non-null   object 
 3   Age              8068 non-null   int64  
 4   Graduated        7990 non-null   object 
 5   Profession       7944 non-null   object 
 6   Work_Experience  7239 non-null   float64
 7   Spending_Score   8068 non-null   object 
 8   Family_Size      7733 non-null   float64
 9   Var_1            7992 non-null   object 
 10  Segmentation     8068 non-null   object 
dtypes: float64(2), int64(2), object(7)
memory usage: 693.5+ KB


In [47]:
target_col = 'Segmentation'

In [48]:
X = raw_df.drop(columns=target_col)
y = LabelEncoder().fit_transform(raw_df[target_col])

In [49]:
X_train, X_test, y_train, y_test = train_test_split(raw_df, y,
    test_size=0.2,
    random_state=42,
    stratify=y                                                
    )

In [55]:
def gender_to_binary(x):
    return (x == 'Male').astype(int)

def yes_no_to_binary(x):
    return (x == 'Yes').astype(int)

def spending_to_ordinal(x):
    mapping = {'Low': 0, 'Average': 1, 'High': 2}
    return x.iloc[:, 0].map(mapping).astype(float).to_frame()

gender_transformer = FunctionTransformer(
    gender_to_binary,
    feature_names_out='one-to-one'
)

yes_or_no_transformer = FunctionTransformer(
    yes_no_to_binary,
    feature_names_out='one-to-one'
)

spending_transformer = FunctionTransformer(
    spending_to_ordinal, 
    feature_names_out='one-to-one'
)

numeric_transformer = StandardScaler()

profession_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

In [56]:
num_cols = ['Age', 'Work_Experience', 'Family_Size']
bin_cols_gender = ['Gender']
bin_cols_yesno = ['Ever_Married', 'Graduated']
ordinal_cols = ['Spending_Score']
ohe_cols = ['Profession', 'Var_1']

In [57]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

ordinal_pipeline = Pipeline([
    ('encoder', FunctionTransformer(spending_to_ordinal, feature_names_out='one-to-one')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, num_cols),
        ('gender', gender_transformer, bin_cols_gender),
        ('yesno', yes_or_no_transformer, bin_cols_yesno),
        ('spend', spending_transformer, ordinal_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ohe_cols),
    ],
    remainder='drop'
)

In [58]:
model = Pipeline([
    ('preprocess', preprocessor),
    ('clf', LogisticRegression(
        max_iter=1000,
        solver='lbfgs'
    ))
])

In [59]:
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('gender', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformer

In [63]:
y_pred = model.predict(X_test)

In [64]:
accuracy_score(y_test, y_pred)

0.5148698884758365

In [66]:
print(classification_report(
    y_test,
    y_pred,
    target_names=['A', 'B', 'C', 'D']
))

              precision    recall  f1-score   support

           A       0.41      0.44      0.43       394
           B       0.41      0.21      0.28       372
           C       0.50      0.61      0.55       394
           D       0.64      0.75      0.69       454

    accuracy                           0.51      1614
   macro avg       0.49      0.50      0.49      1614
weighted avg       0.50      0.51      0.50      1614



**Завдання 2. Важливо уважно прочитати все формулювання цього завдання до кінця!**

Застосуйте методи ресемплингу даних SMOTE та SMOTE-Tomek з бібліотеки imbalanced-learn до тренувальної вибірки. В результаті у Вас має вийти 2 тренувальних набори: з апсемплингом зі SMOTE, та з ресамплингом з SMOTE-Tomek.

Увага! В нашому наборі даних є як категоріальні дані, так і звичайні числові. Базовий SMOTE не буде правильно працювати з категоріальними даними, але є його модифікація, яка буде. Тому в цього завдання є 2 виконання

  1. Застосувати SMOTE базовий лише на НЕкатегоріальних ознаках.

  2. Переглянути інформацію про метод [SMOTENC](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTENC.html#imblearn.over_sampling.SMOTENC) і використати цей метод в цій задачі. За цей спосіб буде +3 бали за це завдання і він рекомендований для виконання.

  **Підказка**: аби скористатись SMOTENC треба створити змінну, яка містить індекси ознак, які є категоріальними (їх номер серед колонок) і передати при ініціації екземпляра класу `SMOTENC(..., categorical_features=cat_feature_indeces)`.
  
  Ви також можете розглянути варіант використання варіації SMOTE, який працює ЛИШЕ з категоріальними ознаками [SMOTEN](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTEN.html)

In [90]:
# Perform random sampling
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

smote = SMOTE(random_state=0)
X_train_smote, y_train_smote = smote.fit_resample(X_train_proc, y_train)

In [91]:
smote_tomek = SMOTETomek(random_state=0)
X_train_st, y_train_st = smote_tomek.fit_resample(X_train_proc, y_train)

In [122]:
X_smotenc = X_train.drop(columns=target_col).copy()

X_smotenc['Work_Experience'] = X_smotenc['Work_Experience'].fillna(X_smotenc['Work_Experience'].median())
X_smotenc['Family_Size'] = X_smotenc['Family_Size'].fillna(X_smotenc['Family_Size'].median())

In [123]:
for col in ['Ever_Married', 'Graduated', 'Profession', 'Var_1']:
    X_smotenc[col] = X_smotenc[col].fillna(X_smotenc[col].mode(dropna=True)[0])

In [129]:
list(enumerate(X_smotenc.columns))

[(0, 'ID'),
 (1, 'Gender'),
 (2, 'Ever_Married'),
 (3, 'Age'),
 (4, 'Graduated'),
 (5, 'Profession'),
 (6, 'Work_Experience'),
 (7, 'Spending_Score'),
 (8, 'Family_Size'),
 (9, 'Var_1')]

In [125]:
cat_feature_indeces = [1, 2, 4, 5, 7, 9]

In [126]:
smotenc = SMOTENC(random_state=42, categorical_features=cat_feature_indeces)

In [127]:
X_train_smotenc, y_train_smotenc = smotenc.fit_resample(X_smotenc, y_train)

In [128]:
X_train_smotenc

,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1
0,465905,Female,No,32,Yes,Artist,9.000000,Low,1.000000,Cat_6
1,462903,Male,Yes,72,Yes,Entertainment,1.000000,Average,2.000000,Cat_6
2,467901,Female,No,33,Yes,Entertainment,1.000000,Low,4.000000,Cat_6
3,463613,Female,Yes,48,Yes,Artist,0.000000,Average,6.000000,Cat_6
4,459859,Female,Yes,28,No,Doctor,9.000000,Low,1.000000,Cat_7
...,...,...,...,...,...,...,...,...,...,...
7251,459113,Male,Yes,71,Yes,Artist,0.940230,High,2.940230,Cat_6
7252,461894,Female,Yes,38,Yes,Artist,2.122629,Average,2.849052,Cat_6
7253,463145,Male,Yes,42,Yes,Artist,3.717189,Average,3.223660,Cat_6
7254,466599,Female,Yes,43,Yes,Artist,6.146306,Average,2.000000,Cat_6


In [134]:
np.bincount(y_train)

array([1578, 1486, 1576, 1814])

In [135]:
np.bincount(y_train_smotenc)

array([1814, 1814, 1814, 1814])

**Завдання 3**.
  1. Навчіть модель логістичної регресії з використанням стратегії One-vs-Rest з логістичною регресією на оригінальних даних, збалансованих з SMOTE, збалансованих з Smote-Tomek.  
  2. Виміряйте якість кожної з натренованих моделей використовуючи `sklearn.metrics.classification_report`.
  3. Напишіть, яку метрику ви обрали для порівняння моделей.
  4. Яка модель найкраща?
  5. Якщо немає суттєвої різниці між моделями - напишіть свою гіпотезу, чому?

In [95]:
# Логістична регресія зі стратегією one-vs-rest (OvR)
log_reg = LogisticRegression(solver='liblinear')
model = OneVsRestClassifier(log_reg)

ovr_model = Pipeline([
    ('preprocess', preprocessor),
    ('clf', model)
])

ovr_model.fit(X_train, y_train)
ovr_predictions = ovr_model.predict(X_test)

print("=== OvR ===")
print(classification_report(y_test, ovr_predictions))

=== OvR ===
              precision    recall  f1-score   support

           0       0.42      0.45      0.43       394
           1       0.42      0.15      0.22       372
           2       0.49      0.64      0.56       394
           3       0.64      0.76      0.70       454

    accuracy                           0.52      1614
   macro avg       0.49      0.50      0.48      1614
weighted avg       0.50      0.52      0.49      1614



In [137]:
X_test_features = X_test.drop(columns=[target_col], errors="ignore").copy()

X_train_sm_proc = preprocessor.fit_transform(X_train_smotenc)
X_test_proc     = preprocessor.transform(X_test_features)

log_reg = LogisticRegression(solver="liblinear", max_iter=2000)
ovr = OneVsRestClassifier(log_reg)

ovr.fit(X_train_sm_proc, y_train_smotenc)
y_pred = ovr.predict(X_test_proc)

print("=== SMOTENC + OvR LogisticRegression ===")
print(classification_report(y_test, y_pred))

=== SMOTENC + OvR LogisticRegression ===
              precision    recall  f1-score   support

           0       0.42      0.48      0.45       394
           1       0.40      0.23      0.29       372
           2       0.50      0.58      0.54       394
           3       0.66      0.72      0.69       454

    accuracy                           0.52      1614
   macro avg       0.50      0.50      0.49      1614
weighted avg       0.50      0.52      0.50      1614



In [93]:
log_reg_smote = LogisticRegression(solver="liblinear", max_iter=2000)
ovr_smote = OneVsRestClassifier(log_reg_smote)

ovr_smote.fit(X_train_smote, y_train_smote)
pred_smote = ovr_smote.predict(X_test_proc)

print("=== SMOTE OvR ===")
print(classification_report(y_test, pred_smote))

=== SMOTE OvR ===
              precision    recall  f1-score   support

           0       0.42      0.47      0.44       394
           1       0.41      0.21      0.28       372
           2       0.49      0.61      0.54       394
           3       0.66      0.71      0.69       454

    accuracy                           0.51      1614
   macro avg       0.49      0.50      0.49      1614
weighted avg       0.50      0.51      0.50      1614



In [94]:
log_reg_st = LogisticRegression(solver="liblinear", max_iter=2000)
ovr_st = OneVsRestClassifier(log_reg_st)

ovr_st.fit(X_train_st, y_train_st)
pred_st = ovr_st.predict(X_test_proc)

print("=== SMOTE-Tomek OvR ===")
print(classification_report(y_test, pred_st))

=== SMOTE-Tomek OvR ===
              precision    recall  f1-score   support

           0       0.41      0.48      0.44       394
           1       0.40      0.19      0.26       372
           2       0.49      0.61      0.54       394
           3       0.68      0.71      0.69       454

    accuracy                           0.51      1614
   macro avg       0.49      0.50      0.48      1614
weighted avg       0.50      0.51      0.49      1614



In [ ]:
# Для порівняння якості моделі я обрав метрику f1 score оскільки задача є багатокласовою

In [ ]:
# Найкращою моделлю є SMOTENC OvR, оскільки показує найкращий f1 score для проблемного міноритарного класу (B)

In [ ]:
# Між моделями немає суттєвої різниці, оскільки ресемплінг та балансування класів не створює якісно нової інформації про класи, а лише балансує їх кількість.